# Task 1: Exploratory Data Analysis (EDA) on Retail Sales Data

**Objective:** Explore retail transactions to understand sales trends, customer demographics, product-category performance, and business opportunities.

**Tools:** Python, pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook.

## Dataset
This notebook uses the Kaggle **Retail Sales Dataset** (1,000 transactions). It contains transaction date, customer gender and age, product category, quantity, unit price, and total amount.

- Kaggle dataset: https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset
- Public CSV mirror used by this notebook: https://raw.githubusercontent.com/TeeshaMahesh/eda-retail-sales-dataset/main/retail_sales_dataset.csv

**Important data limitation:** The selected dataset has a `Product Category` field but does not contain individual product names or product IDs. Therefore, an honest top-10 individual-product ranking cannot be produced. This notebook instead ranks the available product categories by units sold and revenue, and clearly documents this limitation rather than inventing product names.

The dataset has one row per transaction. `Total Amount` is treated as sales revenue. The currency is not specified in the source, so values are shown without a currency symbol.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)

## 2. Load the dataset

In [ ]:
# Load the CSV from the public dataset mirror.
# An internet connection is required for this first download.
DATA_URL = "https://raw.githubusercontent.com/TeeshaMahesh/eda-retail-sales-dataset/main/retail_sales_dataset.csv"

df = pd.read_csv(DATA_URL)
display(df.head())

## 3. Initial inspection: shape, data types, and missing values

In [ ]:
print("Dataset shape (rows, columns):", df.shape)
print("\\nColumn data types:")
print(df.dtypes)
print("\\nDataset information:")
df.info()
print("\\nMissing values in each column:")
display(df.isna().sum().to_frame("Missing Values"))
print("\\nDuplicate rows:", df.duplicated().sum())

### Observation
The inspection shows the number of transactions and fields, each column's data type, and whether any values are missing or duplicated. The `Date` field should be converted to a datetime type before time-series analysis. The checks below also verify the cleaned data.

## 4. Data cleaning and preparation

In [ ]:
# Work on a copy so the original imported data remains unchanged.
df = df.copy()
df.columns = df.columns.str.strip()

# Convert date and numeric fields to appropriate data types.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
numeric_cols = ["Age", "Quantity", "Price per Unit", "Total Amount"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove exact duplicate rows and rows missing essential analysis fields.
before = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["Date", "Age", "Gender", "Product Category",
                       "Quantity", "Price per Unit", "Total Amount"])
print("Rows removed during cleaning:", before - len(df))
print("Cleaned dataset shape:", df.shape)
print("Missing values after cleaning:")
display(df.isna().sum().to_frame("Missing Values"))

# Create time and demographic helper columns.
df["Month"] = df["Date"].dt.to_period("M")
df["Quarter"] = df["Date"].dt.to_period("Q")
df["Age Group"] = pd.cut(
    df["Age"],
    bins=[0, 19, 29, 39, 49, 59, 120],
    labels=["Under 20", "20-29", "30-39", "40-49", "50-59", "60+"],
    right=True
)
display(df.head())

### Observation
The date and numeric columns are converted to suitable data types. Exact duplicate records and rows missing essential analysis fields are removed. Month, quarter, and age-group columns are added to support the analysis. Review the printed row counts to see whether cleaning changed the dataset.

## 5. Descriptive statistics: mean, median, mode, and standard deviation

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
summary = pd.DataFrame({
    "Mean": df[numeric_cols].mean(),
    "Median": df[numeric_cols].median(),
    "Mode": df[numeric_cols].mode().iloc[0],
    "Standard Deviation": df[numeric_cols].std()
})
display(summary.round(2))

### Observation
Mean and median describe the centre of each numeric variable, mode shows its most frequent value, and standard deviation measures spread. A large difference between mean and median can indicate skewness or unusually high transaction values. `Transaction ID` is an identifier, so its statistics are not business performance measures.

## 6. Time-series analysis: monthly sales trend

In [ ]:
monthly_sales = df.set_index("Date").resample("MS")["Total Amount"].sum()

plt.figure(figsize=(12, 5))
plt.plot(monthly_sales.index, monthly_sales.values, marker="o", linewidth=2)
plt.title("Monthly Retail Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Highest-sales month:", monthly_sales.idxmax().strftime("%B %Y"),
      "| Sales:", round(monthly_sales.max(), 2))
print("Lowest-sales month:", monthly_sales.idxmin().strftime("%B %Y"),
      "| Sales:", round(monthly_sales.min(), 2))

### Observation
The line chart shows how total sales change from month to month. The printed highest- and lowest-sales months identify the strongest and weakest periods in this dataset. A business can compare these periods with promotions, holidays, stock availability, and marketing activity before deciding whether the pattern is seasonal.

## 7. Quarterly sales trend

In [ ]:
quarterly_sales = df.groupby("Quarter")["Total Amount"].sum()

plt.figure(figsize=(10, 5))
plt.plot(quarterly_sales.index.astype(str), quarterly_sales.values,
         marker="o", linewidth=2)
plt.title("Quarterly Retail Sales Trend")
plt.xlabel("Quarter")
plt.ylabel("Total Sales")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

display(quarterly_sales.to_frame("Total Sales").round(2))

### Observation
Quarterly totals smooth out some month-to-month variation and make broader changes easier to compare. The table provides exact sales totals for each quarter. Use this view to plan quarterly budgets and inventory, while remembering that the dataset covers a limited time period.

## 8. Customer demographics: age-group distribution

In [ ]:
age_counts = df["Age Group"].value_counts().sort_index()

plt.figure(figsize=(9, 5))
sns.barplot(x=age_counts.index.astype(str), y=age_counts.values)
plt.title("Customer Transaction Count by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()

display(age_counts.to_frame("Transactions"))

### Observation
This chart compares the number of transactions made by each age group. It describes transaction distribution, not necessarily the number of unique customers in each group. The largest group may be a useful audience for customer engagement, while smaller groups can be investigated for unmet needs.

## 9. Customer demographics: gender breakdown

In [ ]:
gender_counts = df["Gender"].value_counts()

plt.figure(figsize=(7, 5))
plt.pie(gender_counts.values, labels=gender_counts.index, autopct="%1.1f%%",
        startangle=90)
plt.title("Transaction Share by Gender")
plt.tight_layout()
plt.show()

display(gender_counts.to_frame("Transactions"))

### Observation
The pie chart shows the share of recorded transactions by gender. This is a transaction share, not a unique-customer share or proof of different preferences. Compare sales value by gender as well before making marketing decisions.

## 10. Product analysis: best-selling available product groups

In [ ]:
# The source has no individual product name or SKU.
# Rank the available product categories by total units sold instead.
category_units = (df.groupby("Product Category")["Quantity"]
                    .sum().sort_values(ascending=False))

plt.figure(figsize=(8, 5))
sns.barplot(x=category_units.index, y=category_units.values)
plt.title("Product Categories Ranked by Units Sold")
plt.xlabel("Product Category")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.show()

display(category_units.to_frame("Units Sold"))

print("Data limitation: individual product names/IDs are not present.")
print("A top-10 individual-product ranking cannot be calculated from this file.")

### Observation
The chart ranks the product categories by units sold. These are category-level groups, not individual products. The dataset contains only three product categories and no product name or SKU, so a top-10 product list would be misleading. To complete that exact subtask, obtain a dataset containing individual product identifiers or product names.

## 11. Revenue by product category

In [ ]:
category_revenue = (df.groupby("Product Category")["Total Amount"]
                      .sum().sort_values(ascending=False))

plt.figure(figsize=(8, 5))
sns.barplot(x=category_revenue.index, y=category_revenue.values)
plt.title("Revenue by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Total Revenue")
plt.tight_layout()
plt.show()

display(category_revenue.to_frame("Revenue").round(2))
print("Highest-revenue category:", category_revenue.idxmax(),
      "| Revenue:", round(category_revenue.max(), 2))

### Observation
This chart identifies which category contributes the most total revenue. The category with the highest revenue is not automatically the most profitable because the dataset does not include product costs or profit. Use revenue alongside margin and inventory data before making purchasing decisions.

## 12. Correlation heatmap between numerical variables

In [ ]:
corr_cols = ["Age", "Quantity", "Price per Unit", "Total Amount"]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f",
            linewidths=0.5, vmin=-1, vmax=1)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

display(corr_matrix.round(2))

### Observation
Correlation ranges from -1 to +1. Positive values indicate variables tend to increase together; negative values indicate an inverse relationship. Correlation does not prove causation. In particular, transaction amount is calculated from quantity and unit price, so a relationship with those fields is expected by construction.

## 13. Additional visualisation: average transaction value by age group

In [ ]:
age_spend = (df.groupby("Age Group", observed=False)["Total Amount"]
              .mean().dropna().sort_index())

plt.figure(figsize=(9, 5))
sns.barplot(x=age_spend.index.astype(str), y=age_spend.values)
plt.title("Average Transaction Value by Customer Age Group")
plt.xlabel("Age Group")
plt.ylabel("Average Transaction Value")
plt.tight_layout()
plt.show()

display(age_spend.to_frame("Average Transaction Value").round(2))
print("Highest average transaction value age group:", age_spend.idxmax(),
      "| Average:", round(age_spend.max(), 2))

### Observation
This chart compares average transaction value across age groups, which can reveal a different pattern from transaction counts alone. A group with fewer transactions may still have a higher average basket value. Treat small differences cautiously and validate them with a larger, longer-term dataset.

## 14. Additional business view: sales by gender and product category

In [ ]:
gender_category = df.pivot_table(
    index="Product Category", columns="Gender",
    values="Total Amount", aggfunc="sum", fill_value=0
)

gender_category.plot(kind="bar", figsize=(9, 5))
plt.title("Revenue by Product Category and Gender")
plt.xlabel("Product Category")
plt.ylabel("Total Revenue")
plt.xticks(rotation=0)
plt.legend(title="Gender")
plt.tight_layout()
plt.show()

display(gender_category.round(2))

### Observation
The grouped bars compare category revenue across gender groups. Differences can help generate hypotheses for product communication or merchandising, but should not be treated as fixed preferences. Confirm patterns with customer counts, repeat purchases, and additional periods before targeting campaigns.

## 15. Key performance indicators

In [ ]:
total_revenue = df["Total Amount"].sum()
total_transactions = df["Transaction ID"].nunique()
total_units = df["Quantity"].sum()
average_transaction_value = df.groupby("Transaction ID")["Total Amount"].sum().mean()
unique_customers = df["Customer ID"].nunique()

kpis = pd.Series({
    "Total Revenue": total_revenue,
    "Unique Transactions": total_transactions,
    "Units Sold": total_units,
    "Average Transaction Value": average_transaction_value,
    "Unique Customers": unique_customers
})
display(kpis.to_frame("Value").round(2))

## 16. Conclusion and actionable business recommendations

Use the actual outputs above to validate these recommendations before submitting:

1. **Plan inventory using category demand:** Allocate replenishment and shelf space according to the categories with the highest unit sales, while monitoring stock-outs and category revenue.
2. **Plan promotions around sales trends:** Compare the strongest and weakest months/quarters. Test targeted promotions in weaker periods and measure whether they increase revenue without reducing margins.
3. **Use customer segments carefully:** Review age-group transaction counts and average transaction values together. Tailor campaigns only where a meaningful difference is visible; avoid assuming gender or age alone determines preferences.
4. **Track basket value:** Monitor average transaction value and test relevant bundles or complementary offers. Measure the change against a baseline before expanding the strategy.
5. **Improve product-level data collection:** Add product name, SKU/product ID, unit cost, discount, and profit fields. This will enable a true top-10 product ranking and profitability analysis.

### Limitations
- The dataset contains 1,000 transactions and a limited date range; findings may not generalise to all retail businesses.
- Individual product names and product IDs are absent, so the top-10 individual-products requirement cannot be fulfilled with this dataset.
- Cost, discount, and profit fields are absent, so profitability and margin cannot be evaluated.
- Currency is not specified in the source.
- Transaction counts are not the same as unique-customer counts.

**Final takeaway:** EDA helps convert raw transaction records into evidence for inventory planning, promotions, customer engagement, and better data collection. Recommendations should be checked against the computed results and additional business data before implementation.


## 17. Reproducibility checklist
- [ ] Run all cells from top to bottom.
- [ ] Confirm that the dataset loaded successfully.
- [ ] Check the cleaning row counts and missing-value output.
- [ ] Verify that every chart appears and each observation is visible.
- [ ] Review the top category and highest/lowest sales periods from the actual output.
- [ ] Save the completed notebook with outputs (`.ipynb`).
- [ ] If your evaluator strictly requires top 10 individual products, replace/extend the dataset with one containing `Product Name` or `Product ID` and update that section.
